# Classificação de textos humanos e de IA com Naive Bayes

MultinomialNB + TF-IDF, com treino, validação e teste. Use as mesmas divisões do XGBoost. 0 = humano; 1 = IA.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)
from xgboost import XGBClassifier

SEED = 42
DATA_PATH = Path('data/ai_human.csv')
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Coloque o CSV em {DATA_PATH.resolve()}')
df = pd.read_csv(DATA_PATH, usecols=['text', 'generated'])

## Limpeza e prevenção de vazamento

Removemos textos vazios e agrupamos duplicatas por texto com espaços normalizados e letras minúsculas. Textos iguais com rótulos conflitantes são descartados. Isso evita que a mesma entrada apareça no treino e no teste; paráfrases e textos da mesma fonte ainda podem exigir agrupamento adicional.

In [ ]:
df = df.dropna(subset=['text', 'generated']).copy()
df['text'] = df['text'].astype(str).str.strip()
df = df.loc[df['text'].ne('')].copy()
df['generated'] = pd.to_numeric(df['generated'], errors='raise')
if not df['generated'].isin([0, 1]).all():
    raise ValueError('A coluna generated deve conter apenas 0 (humano) e 1 (IA).')
df['generated'] = df['generated'].astype(int)
df['_key'] = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True)
conflicts = df.groupby('_key')['generated'].transform('nunique').gt(1)
print('Linhas com rótulos conflitantes removidas:', int(conflicts.sum()))
df = df.loc[~conflicts].drop_duplicates('_key').drop(columns='_key')
if set(df['generated'].unique()) != {0, 1} or df['generated'].value_counts().min() < 10:
    raise ValueError('São necessários pelo menos 10 textos distintos de cada classe.')
print('Textos disponíveis:', len(df))
print(df['generated'].value_counts().rename(index={0: 'Humano', 1: 'IA'}))

In [ ]:
# 60% treino, 20% validação e 20% teste, mantendo a proporção das classes.
X_dev, X_test_text, y_dev, y_test = train_test_split(
    df['text'], df['generated'], test_size=0.2,
    stratify=df['generated'], random_state=SEED)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)

# O vocabulário é aprendido somente no treino.
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000,
                        min_df=2, max_df=1.0, sublinear_tf=True,
                        dtype=np.float32)
X_train = tfidf.fit_transform(X_train_text)
X_val = tfidf.transform(X_val_text)
X_test = tfidf.transform(X_test_text)
print('Matrizes (textos, características):', X_train.shape, X_val.shape, X_test.shape)

## Treinamento

Escolhemos alpha entre 0.1, 0.5, 1 e 2 usando ROC-AUC na validação. O conjunto de teste não participa da escolha.

Referência: https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html

In [ ]:
from sklearn.naive_bayes import MultinomialNB

candidates = []
for alpha in (0.1, 0.5, 1.0, 2.0):
    candidate = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    auc = roc_auc_score(y_val, candidate.predict_proba(X_val)[:, 1])
    candidates.append((auc, candidate))
    print(f'alpha={alpha}: ROC-AUC validação={auc:.4f}')
model = max(candidates, key=lambda item: item[0])[1]
print('Melhor alpha:', model.alpha)

In [ ]:
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
print(f'Acurácia: {accuracy_score(y_test, predictions):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, probabilities):.4f}')
print(classification_report(y_test, predictions, labels=[0, 1],
                            target_names=['Humano', 'IA'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, labels=[0, 1], display_labels=['Humano', 'IA'], cmap='Blues')
plt.title('Naive Bayes — teste reservado')
plt.tight_layout()
plt.show()

In [ ]:
# Salva os dois componentes para reutilizar exatamente o mesmo vocabulário.
Path('models').mkdir(exist_ok=True)
joblib.dump({'vectorizer': tfidf, 'model': model, 'threshold': 0.5,
             'labels': {0: 'Humano', 1: 'IA'}}, 'models/naive_bayes.joblib')

def classificar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError('Informe um texto não vazio.')
    probability = float(model.predict_proba(tfidf.transform([texto]))[0, 1])
    return {'classe': 'IA' if probability >= 0.5 else 'Humano',
            'pontuacao_ia': round(probability, 4)}

# Substitua por um texto no idioma da base.
classificar_texto('Replace this sentence with a text you want to evaluate.')

## Interpretação

A pontuação não é uma probabilidade calibrada de autoria. Avalie outros idiomas, temas e fontes antes de generalizar. Para treinar ambos e gerar uma comparação com as mesmas divisões, execute o notebook comparar_modelos.ipynb.